# Pipeline limpio de analisis, enriquecimiento, modelo ML y dashboard

In [ ]:
import sys
import os
import re
import json
import unicodedata
from collections import Counter, defaultdict

import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath('..'))
from helpers.mysql_connections import create_mysql_engine

engine = create_mysql_engine()


## 1. Extraccion del dataset maestro

In [ ]:
q_master = """
SELECT
    r.id AS result_id,
    r.campaign_id,
    r.user_id,
    r.r_id,
    r.email,
    r.first_name,
    r.last_name,
    r.position,
    r.status AS result_status,
    r.ip,
    r.latitude,
    r.longitude,
    r.send_date AS result_send_date,
    r.reported,
    r.modified_date,

    c.name AS campaign_name,
    c.status AS campaign_status,
    c.created_date AS campaign_created_date,
    c.completed_date AS campaign_completed_date,
    c.template_id,
    c.page_id,
    c.smtp_id,
    c.url AS campaign_url,
    c.launch_date,

    t.name AS template_name,
    t.subject AS template_subject,
    t.text AS template_text,
    t.modified_date AS template_modified_date,

    e.from_address,
    e.url AS request_url,
    e.template_id AS er_template_id,
    e.page_id AS er_page_id,

    m.send_date AS mail_send_date,
    m.send_attempt,
    m.processing

FROM gophish.results r
LEFT JOIN gophish.campaigns c
    ON r.campaign_id = c.id
LEFT JOIN gophish.templates t
    ON c.template_id = t.id
LEFT JOIN gophish.email_requests e
    ON r.r_id = e.r_id
LEFT JOIN gophish.mail_logs m
    ON r.r_id = m.r_id
"""

df_master = pd.read_sql(q_master, engine)

print(df_master.shape)
print(df_master.columns.tolist())
df_master.head()

df_master = pd.read_sql(q_master, engine)
df_master.head()

## 2. Limpieza de texto y carga del diccionario

In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower().strip()
    text = ''.join(
        c for c in unicodedata.normalize('NFKD', text)
        if not unicodedata.combining(c)
    )
    text = re.sub(r'[\r\n\t]+', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df = df_master.copy()

subject_col = "template_subject" if "template_subject" in df.columns else None
text_col = "template_text" if "template_text" in df.columns else None

if subject_col is None and text_col is None:
    raise ValueError("No se encontraron columnas de texto como template_subject o template_text en df")

df["text_full"] = (
    (df[subject_col].fillna("") if subject_col else "") + " " +
    (df[text_col].fillna("") if text_col else "")
).apply(clean_text)

df_cls = pd.read_sql("SELECT * FROM gophish.text_clasification", engine)
df_cls["text_clean"] = df_cls["text"].apply(clean_text)

df[["text_full"]].head(), df_cls.head()


## 3. Palabras comunes derivadas de la base

In [ ]:
SPANISH_STOPWORDS = {
    "de", "la", "el", "y", "en", "para", "con", "por",
    "del", "los", "las", "un", "una", "al", "se", "lo"
}

doc_freq = Counter()
for text in df_cls["text_clean"].fillna(""):
    words = set(str(text).split())
    doc_freq.update(words)

freq_df = pd.DataFrame({
    "word": list(doc_freq.keys()),
    "doc_count": list(doc_freq.values())
})
freq_df["doc_ratio"] = freq_df["doc_count"] / len(df_cls)
freq_df = freq_df.sort_values(["doc_count", "doc_ratio"], ascending=False).reset_index(drop=True)

COMMON_WORDS = set(
    freq_df.loc[
        (freq_df["doc_ratio"] >= 0.04) | (freq_df["word"].str.len() < 3),
        "word"
    ]
).union(SPANISH_STOPWORDS)

COMMON_WORDS = COMMON_WORDS - {
    "credenciales",
    "contrasena",
    "token",
    "verificacion",
    "identidad",
    "banco",
    "tarjeta",
    "acceso",
    "pago"
}

sorted(list(COMMON_WORDS))[:50]


## 4. Matcher y enriquecimiento semantico

In [ ]:
def build_text_matcher(df_cls, common_words):
    patterns = []
    inverted_index = defaultdict(set)

    for idx, row in df_cls.iterrows():
        pattern_text = str(row["text_clean"])
        pattern_words = set(pattern_text.split())
        useful_words = {w for w in pattern_words if w not in common_words}

        item = {
            "id": idx,
            "pattern_text": pattern_text,
            "pattern_words": pattern_words,
            "useful_words": useful_words,
            "context": row["context"],
            "risk": row["risk"],
            "is_urgent": int(row["is_urgent"]),
            "has_credentials": int(row["has_credentials"]),
            "message_type": row["message_type"],
            "original_text": row["text"]
        }

        patterns.append(item)

        for word in useful_words:
            inverted_index[word].add(idx)

    return {
        "patterns": patterns,
        "inverted_index": inverted_index,
        "common_words": common_words
    }


def analyze_text_fast(text, matcher, min_overlap=2, top_k=4):
    text = str(text)
    text_words = set(text.split())

    useful_text_words = {
        w for w in text_words
        if w not in matcher["common_words"]
    }

    candidate_ids = set()
    for word in useful_text_words:
        candidate_ids.update(matcher["inverted_index"].get(word, set()))

    if not candidate_ids:
        return pd.Series({
            "risk_flag": 0,
            "urgent_flag": 0,
            "credentials_flag": 0,
            "context_detected": "None",
            "risk_level": "None",
            "match_score": 0,
            "matched_patterns": [],
            "matched_terms": []
        })

    matched_rows = []
    for idx in candidate_ids:
        item = matcher["patterns"][idx]
        exact_match = item["pattern_text"] in text
        overlap_words = useful_text_words.intersection(item["useful_words"])
        overlap_score = len(overlap_words)

        if not exact_match and overlap_score < min_overlap:
            continue

        local_score = 0
        if exact_match:
            local_score += 4
        local_score += min(overlap_score, 3)

        matched_rows.append({
            "context": item["context"],
            "risk": item["risk"],
            "is_urgent": int(item["is_urgent"]),
            "has_credentials": int(item["has_credentials"]),
            "exact_match": int(exact_match),
            "overlap_score": overlap_score,
            "local_score": local_score,
            "matched_pattern": item["original_text"],
            "matched_terms": list(overlap_words)
        })

    if not matched_rows:
        return pd.Series({
            "risk_flag": 0,
            "urgent_flag": 0,
            "credentials_flag": 0,
            "context_detected": "None",
            "risk_level": "None",
            "match_score": 0,
            "matched_patterns": [],
            "matched_terms": []
        })

    df_match = pd.DataFrame(matched_rows).sort_values(
        ["local_score", "exact_match", "overlap_score"],
        ascending=False
    ).head(top_k)

    best_row = df_match.iloc[0]

    return pd.Series({
        "risk_flag": 1,
        "urgent_flag": int(df_match["is_urgent"].max()),
        "credentials_flag": int(df_match["has_credentials"].max()),
        "context_detected": best_row["context"],
        "risk_level": best_row["risk"],
        "match_score": int(best_row["local_score"]),
        "matched_patterns": list(df_match["matched_pattern"].unique()),
        "matched_terms": sorted(
            list(
                set(term for sublist in df_match["matched_terms"] for term in sublist)
            )
        )
    })


matcher = build_text_matcher(df_cls, COMMON_WORDS)

extra_features = df["text_full"].apply(
    lambda x: analyze_text_fast(x, matcher, min_overlap=2, top_k=4)
)

df_features = pd.concat([df, extra_features], axis=1)

df_features["risk_level"].value_counts(dropna=False)


## 5. Modelo ML final

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import classification_report, accuracy_score

X = df_features[[
    "text_full",
    "risk_flag",
    "urgent_flag",
    "credentials_flag",
    "context_detected",
    "risk_level"
]].copy()

y = df_features["result_status"]

text_col = "text_full"
num_cols = ["risk_flag", "urgent_flag", "credentials_flag"]
cat_cols = ["context_detected", "risk_level"]

preprocessor = ColumnTransformer([
    ("text", TfidfVectorizer(ngram_range=(1, 2), min_df=2), text_col),
    ("num", "passthrough", num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

model = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nReporte:")
print(classification_report(y_test, y_pred))


## 6. Dashboard final

In [ ]:
def add_data_labels(ax, fmt="{:.0f}", fontsize=10, padding=3):
    for p in ax.patches:
        width = p.get_width()
        height = p.get_height()
        if height is not None and height > 0:
            x = p.get_x() + width / 2
            y = height
            ax.annotate(
                fmt.format(height),
                (x, y),
                ha="center",
                va="bottom",
                textcoords="offset points",
                xytext=(0, padding),
                fontsize=fontsize
            )

df_dash = df_features.copy()

total_records = len(df_dash)
status_counts = df_dash["result_status"].value_counts(dropna=False)
risk_counts = df_dash["risk_level"].value_counts(dropna=False)
context_counts = df_dash["context_detected"].value_counts(dropna=False)
risk_flag_counts = df_dash["risk_flag"].value_counts(dropna=False)
urgent_flag_counts = df_dash["urgent_flag"].value_counts(dropna=False)
credentials_flag_counts = df_dash["credentials_flag"].value_counts(dropna=False)

dashboard_summary = pd.DataFrame({
    "metric": [
        "total_records", "email_opened", "clicked_link", "email_sent", "submitted_data",
        "risk_bajo", "risk_medio", "risk_alto", "risk_none",
        "risk_flag_1", "urgent_flag_1", "credentials_flag_1"
    ],
    "value": [
        total_records,
        status_counts.get("Email Opened", 0),
        status_counts.get("Clicked Link", 0),
        status_counts.get("Email Sent", 0),
        status_counts.get("Submitted Data", 0),
        risk_counts.get("Bajo", 0),
        risk_counts.get("Medio", 0),
        risk_counts.get("Alto", 0),
        risk_counts.get("None", 0),
        risk_flag_counts.get(1, 0),
        urgent_flag_counts.get(1, 0),
        credentials_flag_counts.get(1, 0),
    ]
})
dashboard_summary

plt.figure(figsize=(10, 6))
ax = status_counts.plot(kind="bar")
plt.title("Distribucion de estados finales")
plt.xlabel("Estado")
plt.ylabel("Cantidad")
plt.xticks(rotation=45)
add_data_labels(ax)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
ax = risk_counts.plot(kind="bar")
plt.title("Distribucion de niveles de riesgo")
plt.xlabel("Nivel de riesgo")
plt.ylabel("Cantidad")
plt.xticks(rotation=0)
add_data_labels(ax)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
ax = context_counts.plot(kind="bar")
plt.title("Contextos detectados en los mensajes")
plt.xlabel("Contexto")
plt.ylabel("Cantidad")
plt.xticks(rotation=45)
add_data_labels(ax)
plt.tight_layout()
plt.show()

campaign_total = (
    df_dash.groupby(["campaign_id", "campaign_name"])
    .size()
    .reset_index(name="total_events")
    .sort_values("total_events", ascending=False)
)

top_campaigns = campaign_total.head(10)
plt.figure(figsize=(12, 6))
ax = plt.gca()
ax.bar(top_campaigns["campaign_name"], top_campaigns["total_events"])
plt.title("Top 10 campanas por volumen de interacciones")
plt.xlabel("Campana")
plt.ylabel("Cantidad")
plt.xticks(rotation=75)
add_data_labels(ax)
plt.tight_layout()
plt.show()

context_risk_summary = (
    df_dash.groupby(["context_detected", "risk_level"])
    .size()
    .reset_index(name="count")
)

context_risk_pivot = context_risk_summary.pivot(
    index="context_detected",
    columns="risk_level",
    values="count"
).fillna(0)

risk_order = ["Alto", "Medio", "Bajo"]
context_risk_pivot = context_risk_pivot[[c for c in risk_order if c in context_risk_pivot.columns]]

risk_colors = {
    "Alto": "#ff9999",
    "Medio": "#fff59d",
    "Bajo": "#b9f6ca"
}
colors = [risk_colors[col] for col in context_risk_pivot.columns]

ax = context_risk_pivot.plot(kind="bar", figsize=(12, 6), color=colors)
plt.title("Distribucion de riesgo por contexto detectado")
plt.xlabel("Contexto")
plt.ylabel("Cantidad")
plt.xticks(rotation=45)
add_data_labels(ax, fontsize=8, padding=2)
plt.legend(title="risk_level")
plt.tight_layout()
plt.show()


## 7. Persistencia opcional

In [ ]:
# Persistencia opcional

# Guardar dataset enriquecido (serializando columnas tipo lista)
""" df_save = df_features.copy()
for col in ["matched_patterns", "matched_terms"]:
    if col in df_save.columns:
        df_save[col] = df_save[col].apply(
            lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, (list, dict)) else x
        )

# Si la conexion quedo en estado invalido por una operacion previa, recrear engine
engine = create_mysql_engine()

dashboard_summary.to_sql("dashboard_summary", con=engine, if_exists="replace", index=False)
campaign_total.to_sql("dashboard_campaign_total", con=engine, if_exists="replace", index=False)
context_risk_summary.to_sql("dashboard_context_risk", con=engine, if_exists="replace", index=False) """

# Ejemplos opcionales:
# df_save.to_sql("ml_features_results", con=engine, if_exists="replace", index=False)
# import joblib
# joblib.dump(model, "modelo_ml_final.joblib")

""" print("Persistencia completada.") """
